# 🌿 Darukaa Reference Benchmarking Pipeline

**Compare any biodiversity indicator against ecoregion-specific reference benchmarks.**

This notebook runs the full pipeline:
1. Clone the repo & install dependencies
2. Authenticate Google Earth Engine
3. Upload your KML site file
4. Run Tier 1 (global) + Tier 2 (contemporary) benchmarking
5. Download the scorecard (JSON + CSV)

**Methodology:**
- McElderry et al. (2024) DOI:10.32942/X2689N — SEED reference selection
- McNellie et al. (2020) DOI:10.1111/gcb.15383 — Contemporary reference states
- Yen et al. (2019) DOI:10.1002/eap.1970 — Biodiversity benchmarks

---

## 1. Setup — Clone repo & install

In [ ]:
# ── Setup: Clone or update repo ──────────────────────────
import os
os.chdir("/content")  # Always start from a safe location

REPO_DIR = "/content/reference-benchmarking"
CODE_DIR = f"{REPO_DIR}/darukaa_reference_v0.1.0"

if os.path.exists(REPO_DIR):
    os.chdir(CODE_DIR)
    !git pull origin main
    print("✓ Pulled latest changes")
else:
    !git clone https://@github.com/G-auravSingh/reference-benchmarking.git {REPO_DIR}
    os.chdir(CODE_DIR)
    print("✓ Cloned fresh")

!pip install -e . --force-reinstall -q 2>&1 | tail -1

import sys
cleared = [k for k in sys.modules if 'darukaa' in k]
for m in cleared:
    del sys.modules[m]

os.chdir("/content")
print(f"✓ Installed from {CODE_DIR}, cleared {len(cleared)} cached modules")

remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 22 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (22/22), 7.18 KiB | 816.00 KiB/s, done.
From https://github.com/G-auravSingh/reference-benchmarking
 * branch            main       -> FETCH_HEAD
   34f58f1..5c9624d  main       -> origin/main
Updating 34f58f1..5c9624d
Fast-forward
 .../darukaa_reference/config.py                    |   5 +
 .../darukaa_reference/indicators/__init__.py       |  60 ++++---
 .../darukaa_reference/reference.py                 |  26 +--
 .../notebooks/run_pipeline.ipynb                   | 199 +++++++++++----------
 4 files changed, 159 insertions(+), 131 deletions(-)
✓ Pulled latest changes
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
✓ Installed from /content/refer

## 2. Authenticate Google Earth Engine

Run the cell below — it will show a link. Click it, sign in with your Google account, copy the token back.

In [ ]:
import ee

# Colab has a built-in GEE authenticator
ee.Authenticate()

# Initialize with your GEE project
# Replace with your actual project ID
GEE_PROJECT = "gaurav-singh-007"  # @param {type:"string"}

ee.Initialize(project=GEE_PROJECT)
print(f"✓ GEE initialised with project: {GEE_PROJECT}")

✓ GEE initialised with project: gaurav-singh-007


## 3. Upload your KML file

Run the cell below to upload your project site KML/KMZ/GeoJSON file.

In [ ]:
from google.colab import files

print("Upload your KML/KMZ/GeoJSON file:")
uploaded = files.upload()

# Get the filename
SITE_FILE = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {SITE_FILE} ({len(uploaded[SITE_FILE])/1024:.1f} KB)")

Upload your KML/KMZ/GeoJSON file:


Saving Paglam_CR_Demarcation2025.kmz to Paglam_CR_Demarcation2025.kmz

✓ Uploaded: Paglam_CR_Demarcation2025.kmz (1.6 KB)


## 4. Configure the pipeline

Adjust parameters below as needed.

In [ ]:
# ── Configuration ─────────────────────────────────────────

# Which indicators to run?
# Options: "ndvi", "lst", "bii", "eii", "ghm", "msa_globio4", "seed"
# Leave empty for ALL GEE-based indicators
INDICATORS = ["ndvi", "bii", "eii", "ghm", "lst"]  # @param

# Tier 2 reference selection
BUFFER_KM = 100        # Search radius for reference patches (km)
HMI_PERCENTILE = 5     # Top N% least disturbed = reference

# NDVI year
NDVI_YEAR = 2025       # @param {type:"integer"}

# ─────────────────────────────────────────────────────────
print(f"Indicators: {INDICATORS}")
print(f"Tier 2: buffer={BUFFER_KM}km, HMI top {HMI_PERCENTILE}%")
print(f"NDVI year: {NDVI_YEAR}")

Indicators: ['ndvi', 'bii', 'eii', 'ghm', 'lst']
Tier 2: buffer=100km, HMI top 5%
NDVI year: 2025


## 5. Run the pipeline

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

from darukaa_reference.config import Config
from darukaa_reference.registry import IndicatorRegistry
from darukaa_reference.indicators import create_default_registry
from darukaa_reference.pipeline import Pipeline

# Build config programmatically (no YAML needed in Colab)
config = Config(
    gee_project=GEE_PROJECT,
    ecoregion_source="gee",
    reference_buffer_km=BUFFER_KM,
    hmi_percentile_threshold=HMI_PERCENTILE,
    ndvi_year=NDVI_YEAR,
    lst_year=NDVI_YEAR,
    elevation_band_m=300.0,
    bii_gee_asset="projects/gaurav-singh-007/assets/bii-2020_v2-1-1",  # ← your BII
    raster_paths={"bii": "/content/bii-2020_v2-1-1.tif"},  # local fallback
    output_dir="./output",
    output_format="both",
    enabled_indicators=INDICATORS,
)
# Create registry
registry = create_default_registry()

# Build and run pipeline
pipeline = Pipeline(config, registry)
report = pipeline.run(
    site_path=SITE_FILE,
    output_path="./output/benchmark_scorecard",
)

## 6. View results

In [ ]:
import pandas as pd

# Load scorecard as DataFrame
df = pd.DataFrame(report["scorecard"])

# Display key columns
display_cols = [
    "site_id", "display_name", "pillar", "site_value",
    "tier1_reference", "tier1_intactness",
    "tier2_reference", "tier2_intactness",
    "hedges_g", "permutation_p", "interpretation",
]
existing_cols = [c for c in display_cols if c in df.columns]

print(f"\n{'='*70}")
print(f"BENCHMARK SCORECARD: {len(df)} indicator-site pairs")
print(f"{'='*70}\n")

df[existing_cols].style.format({
    "site_value": "{:.4f}",
    "tier1_reference": "{:.4f}",
    "tier1_intactness": "{:.1%}",
    "tier2_reference": "{:.4f}",
    "tier2_intactness": "{:.1%}",
    "hedges_g": "{:.3f}",
    "permutation_p": "{:.4f}",
}, na_rep="—")


BENCHMARK SCORECARD: 5 indicator-site pairs



,site_id,display_name,pillar,site_value,tier1_reference,tier1_intactness,tier2_reference,tier2_intactness,hedges_g,permutation_p,interpretation
0,Paglam_CR_Demarcation2025_0000,Vegetation Structure (NDVI),1,0.6282,0.6753,93.0%,0.7294,86.1%,—,—,Moderate intactness (86.1% of Tier 2 benchmark)
1,Paglam_CR_Demarcation2025_0000,Biodiversity Intactness Index,3,0.4055,0.4572,88.7%,0.6056,67.0%,—,—,Degraded (67.0% of Tier 2 benchmark)
2,Paglam_CR_Demarcation2025_0000,Ecosystem Integrity Index,1,0.2604,0.3655,71.2%,0.5109,51.0%,—,—,Degraded (51.0% of Tier 2 benchmark)
3,Paglam_CR_Demarcation2025_0000,Global Human Modification,4,0.3366,0.3889,100.0%,—,—,—,—,Insufficient data for interpretation
4,Paglam_CR_Demarcation2025_0000,Land Surface Temperature,1,23.5373,23.8631,100.0%,23.7932,100.0%,—,—,Near-reference condition (100.0% of Tier 2 benchmark)


In [ ]:
# Pillar summary
if report.get("pillar_summary"):
    print("\nPILLAR INTACTNESS SUMMARY")
    print("-" * 60)
    for ps in report["pillar_summary"]:
        print(
            f"  Pillar {ps['pillar']}: {ps['pillar_name']:30s} | "
            f"Mean: {ps['mean_intactness']:.1%} | "
            f"Range: [{ps['min_intactness']:.1%}, {ps['max_intactness']:.1%}]"
        )


PILLAR INTACTNESS SUMMARY
------------------------------------------------------------
  Pillar 1: Ecosystem Condition            | Mean: 79.0% | Range: [51.0%, 100.0%]
  Pillar 3: Species/Population Status      | Mean: 67.0% | Range: [67.0%, 67.0%]


## 7. Download results

In [ ]:
from google.colab import files

# Download JSON
files.download("./output/benchmark_scorecard.json")

# Download CSV
files.download("./output/benchmark_scorecard.csv")

print("\n✓ Scorecard downloaded (JSON + CSV)")

---

## (Optional) Add a custom indicator

Adding any new indicator takes just a few lines — no pipeline changes needed.

In [ ]:
# Example: Add CPLAND (connectivity) as a custom indicator

def extract_cpland(geometry, config):
    """
    Your CPLAND extraction logic here.
    Could call Darukaa's internal API, compute from FRAGSTATS, etc.
    """
    # Placeholder — replace with your real implementation
    return {"value": 0.2135, "pixels": None}

registry.register(
    name="cpland",
    display_name="Landscape Connectivity (CPLAND)",
    source_type="api",
    extract_fn=extract_cpland,
    unit="%",
    value_range=(0.0, 100.0),
    citation="McGarigal & Marks (1995). FRAGSTATS. USDA Forest Service.",
    tier2_eligible=True,
    pillar=1,
)

print(f"✓ Registry now has {len(registry)} indicators: {registry.names()}")